In [1]:
import json
import re
from pathlib import Path
from PIL import Image
from paddleocr import PaddleOCR

/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [2]:
IMG_DIR = Path("/root/lorex-ocr-llm-experiments/data/cord_test/images")
OUT_PATH = Path("/root/lorex-ocr-llm-experiments/results/cord_ocr/paddle_ocr_cord_test_filtered.json")

MAX_IMAGES = 100
MAX_SIDE = 2000

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

all_image_paths = sorted(IMG_DIR.glob("*.png"), key=lambda p: int(p.stem))[:MAX_IMAGES]
print("Total candidate images:", len(all_image_paths))

Total candidate images: 100


In [3]:
filtered_image_paths = []
skipped_large = []

for p in all_image_paths:
    img = Image.open(p)
    w, h = img.size
    if max(w, h) > MAX_SIDE:
        skipped_large.append({"source_image": p.name, "size": (w, h)})
    else:
        filtered_image_paths.append(p)

print("Kept:", len(filtered_image_paths))
print("Skipped large:", len(skipped_large))
print("Skipped examples:", skipped_large[:10])

Kept: 83
Skipped large: 17
Skipped examples: [{'source_image': '5.png', 'size': (2304, 4096)}, {'source_image': '14.png', 'size': (1836, 3264)}, {'source_image': '31.png', 'size': (2304, 4096)}, {'source_image': '32.png', 'size': (1836, 3264)}, {'source_image': '36.png', 'size': (2304, 4096)}, {'source_image': '38.png', 'size': (2304, 4096)}, {'source_image': '42.png', 'size': (1836, 3264)}, {'source_image': '43.png', 'size': (2304, 4096)}, {'source_image': '58.png', 'size': (1836, 3264)}, {'source_image': '59.png', 'size': (3024, 4032)}]


In [4]:
paddle_ocr = PaddleOCR(
    lang="en",
    use_textline_orientation=True,
)

/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files al

In [5]:
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def flatten_ocr_text(result) -> str:
    if not result:
        return ""
    page = result[0]
    if isinstance(page, dict) and "rec_texts" in page:
        texts = [t.strip() for t in page["rec_texts"] if isinstance(t, str) and t.strip()]
        return clean_text("\n".join(texts))
    return ""

def load_existing_results(path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_results(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [6]:
results = load_existing_results(OUT_PATH)
done_ids = {rec["source_image"] for rec in results if "source_image" in rec}

print("Already saved:", len(done_ids))
print("Remaining:", len([p for p in filtered_image_paths if p.name not in done_ids]))

Already saved: 0
Remaining: 83


In [7]:
for idx, img_path in enumerate(filtered_image_paths, start=1):
    if img_path.name in done_ids:
        continue

    print(f"[{idx}/{len(filtered_image_paths)}] OCR {img_path.name}")

    try:
        res = paddle_ocr.predict(str(img_path))
        raw_text = flatten_ocr_text(res)

        rec = {
            "source_image": img_path.name,
            "ocr_text": raw_text,
        }

    except Exception as e:
        rec = {
            "source_image": img_path.name,
            "ocr_text": "",
            "error": str(e),
        }

    results.append(rec)
    done_ids.add(img_path.name)

    save_results(results, OUT_PATH)
    print("Saved:", img_path.name, "| total saved:", len(results))

[1/83] OCR 0.png
Saved: 0.png | total saved: 1
[2/83] OCR 1.png
Saved: 1.png | total saved: 2
[3/83] OCR 2.png
Saved: 2.png | total saved: 3
[4/83] OCR 3.png
Saved: 3.png | total saved: 4
[5/83] OCR 4.png
Saved: 4.png | total saved: 5
[6/83] OCR 6.png
Saved: 6.png | total saved: 6
[7/83] OCR 7.png
Saved: 7.png | total saved: 7
[8/83] OCR 8.png
Saved: 8.png | total saved: 8
[9/83] OCR 9.png
Saved: 9.png | total saved: 9
[10/83] OCR 10.png
Saved: 10.png | total saved: 10
[11/83] OCR 11.png
Saved: 11.png | total saved: 11
[12/83] OCR 12.png
Saved: 12.png | total saved: 12
[13/83] OCR 13.png
Saved: 13.png | total saved: 13
[14/83] OCR 15.png
Saved: 15.png | total saved: 14
[15/83] OCR 16.png
Saved: 16.png | total saved: 15
[16/83] OCR 17.png
Saved: 17.png | total saved: 16
[17/83] OCR 18.png
Saved: 18.png | total saved: 17
[18/83] OCR 19.png
Saved: 19.png | total saved: 18
[19/83] OCR 20.png
Saved: 20.png | total saved: 19
[20/83] OCR 21.png
Saved: 21.png | total saved: 20
[21/83] OCR 22.p

In [8]:
print("Final OCR records:", len(results))
results[:2]

Final OCR records: 83


[{'source_image': '0.png',
  'ocr_text': '901016\n-TICKET CP\n2\n60,000\n00,000\nTOTAL DISC $\n-60,000\nTAX\n5.455\nSubtotal\n60.000\nTOTAL\n(Qty2.0060.000\n-EDC CIMB NIAGA NO: XX7730\n60.000'},
 {'source_image': '1.png',
  'ocr_text': 'DAMD\nECRRRIOAE\naiMia\nBIEME\nTRETEMS\nIAHRHDEM\nBAAIR\n1\nH\nJ.STB PROMO\n17500\nY.B.BAT\n46000\nY.BASO PROM\n27500\nTOTAL\n91000\nCASH\n91000\nBT\nATE'}]

### CORD & Tesseract

In [9]:
import pytesseract
from PIL import Image

In [10]:
TESS_OUT_PATH = Path("/root/lorex-ocr-llm-experiments/results/cord_ocr/tesseract_ocr_cord_test_filtered.json")
print(TESS_OUT_PATH)

/root/lorex-ocr-llm-experiments/results/cord_ocr/tesseract_ocr_cord_test_filtered.json


In [11]:
def tesseract_extract_text(img_path):
    img = Image.open(img_path)
    text = pytesseract.image_to_string(img, lang="eng")
    return clean_text(text)

In [12]:
tess_results = load_existing_results(TESS_OUT_PATH)
tess_done_ids = {rec["source_image"] for rec in tess_results if "source_image" in rec}

print("Already saved:", len(tess_done_ids))
print("Remaining:", len([p for p in filtered_image_paths if p.name not in tess_done_ids]))

Already saved: 0
Remaining: 83


In [13]:
for idx, img_path in enumerate(filtered_image_paths, start=1):
    if img_path.name in tess_done_ids:
        continue

    print(f"[{idx}/{len(filtered_image_paths)}] Tesseract {img_path.name}")

    try:
        raw_text = tesseract_extract_text(img_path)

        rec = {
            "source_image": img_path.name,
            "ocr_text": raw_text,
        }

    except Exception as e:
        rec = {
            "source_image": img_path.name,
            "ocr_text": "",
            "error": str(e),
        }

    tess_results.append(rec)
    tess_done_ids.add(img_path.name)

    save_results(tess_results, TESS_OUT_PATH)
    print("Saved:", img_path.name, "| total saved:", len(tess_results))

[1/83] Tesseract 0.png
Saved: 0.png | total saved: 1
[2/83] Tesseract 1.png
Saved: 1.png | total saved: 2
[3/83] Tesseract 2.png
Saved: 2.png | total saved: 3
[4/83] Tesseract 3.png
Saved: 3.png | total saved: 4
[5/83] Tesseract 4.png
Saved: 4.png | total saved: 5
[6/83] Tesseract 6.png
Saved: 6.png | total saved: 6
[7/83] Tesseract 7.png
Saved: 7.png | total saved: 7
[8/83] Tesseract 8.png
Saved: 8.png | total saved: 8
[9/83] Tesseract 9.png
Saved: 9.png | total saved: 9
[10/83] Tesseract 10.png
Saved: 10.png | total saved: 10
[11/83] Tesseract 11.png
Saved: 11.png | total saved: 11
[12/83] Tesseract 12.png
Saved: 12.png | total saved: 12
[13/83] Tesseract 13.png
Saved: 13.png | total saved: 13
[14/83] Tesseract 15.png
Saved: 15.png | total saved: 14
[15/83] Tesseract 16.png
Saved: 16.png | total saved: 15
[16/83] Tesseract 17.png
Saved: 17.png | total saved: 16
[17/83] Tesseract 18.png
Saved: 18.png | total saved: 17
[18/83] Tesseract 19.png
Saved: 19.png | total saved: 18
[19/83] Te

In [14]:
print("Final Tesseract OCR records:", len(tess_results))
tess_results[:2]

Final Tesseract OCR records: 83


[{'source_image': '0.png',
  'ocr_text': '-TICKET CP\n2 60,000 £0,000\n901016\nTOTAL DISC $\nTAX\nSubtotal\nTOTAL caty 2.00 60,000\n= EDC CINB NIAGA No: xx7730 60.000\nee maeh Tur),'},
 {'source_image': '1.png', 'ocr_text': ''}]

In [15]:
import easyocr

In [16]:
easy_reader = easyocr.Reader(['en'], gpu=False)

[2026-04-20 16:55:44,379] [ WARNING] easyocr.py:71 - Using CPU. Note: This module is much faster with a GPU.


In [17]:
EASY_OUT_PATH = Path("/root/lorex-ocr-llm-experiments/results/cord_ocr/easyocr_ocr_cord_test_filtered.json")
print(EASY_OUT_PATH)

/root/lorex-ocr-llm-experiments/results/cord_ocr/easyocr_ocr_cord_test_filtered.json


In [18]:
def easyocr_extract_text(img_path):
    result = easy_reader.readtext(str(img_path), detail=0, paragraph=False)
    text = "\n".join([x.strip() for x in result if str(x).strip()])
    return clean_text(text)

In [19]:
easy_results = load_existing_results(EASY_OUT_PATH)
easy_done_ids = {rec["source_image"] for rec in easy_results if "source_image" in rec}

print("Already saved:", len(easy_done_ids))
print("Remaining:", len([p for p in filtered_image_paths if p.name not in easy_done_ids]))

Already saved: 0
Remaining: 83


In [20]:
for idx, img_path in enumerate(filtered_image_paths, start=1):
    if img_path.name in easy_done_ids:
        continue

    print(f"[{idx}/{len(filtered_image_paths)}] EasyOCR {img_path.name}")

    try:
        raw_text = easyocr_extract_text(img_path)

        rec = {
            "source_image": img_path.name,
            "ocr_text": raw_text,
        }

    except Exception as e:
        rec = {
            "source_image": img_path.name,
            "ocr_text": "",
            "error": str(e),
        }

    easy_results.append(rec)
    easy_done_ids.add(img_path.name)

    save_results(easy_results, EASY_OUT_PATH)
    print("Saved:", img_path.name, "| total saved:", len(easy_results))

[1/83] EasyOCR 0.png


/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12000). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Saved: 0.png | total saved: 1
[2/83] EasyOCR 1.png


/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Saved: 1.png | total saved: 2
[3/83] EasyOCR 2.png


...

Saved: 96.png | total saved: 81
[82/83] EasyOCR 97.png


/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Saved: 97.png | total saved: 82
[83/83] EasyOCR 99.png


/root/lorex-ocr-llm-experiments/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Saved: 99.png | total saved: 83


In [21]:
print("Final EasyOCR records:", len(easy_results))
easy_results[:2]

Final EasyOCR records: 83


[{'source_image': '0.png',
  'ocr_text': '901016\n-TICKET Cp\n60,0oo\nEO,Q00\nTOTAL dis:\n60,000\ntak\n5,455\nSubtota]\n6O,000\nToTaL\n(Oty\n00\n60\n000\nEdC CIMB NIAGA No: xx7730\n60,000'},
 {'source_image': '1.png',
  'ocr_text': 'J.STE\nFfomo\nI7300\nY.B .BaT\n460u0\nY.BaSU\nProh\n27300\nT 0 T AL\n9 1 0 0 0\nC Ash\n9 1 0 0 0'}]

### DOCtr

In [27]:
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

In [28]:
doctr_model = ocr_predictor(pretrained=True)

In [29]:
DOCTR_OUT_PATH = Path("/root/lorex-ocr-llm-experiments/results/cord_ocr/doctr_ocr_cord_test_filtered.json")
print(DOCTR_OUT_PATH)

/root/lorex-ocr-llm-experiments/results/cord_ocr/doctr_ocr_cord_test_filtered.json


In [30]:
def doctr_extract_text(img_path):
    doc = DocumentFile.from_images(str(img_path))
    result = doctr_model(doc)

    lines = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                text = " ".join([word.value for word in line.words])
                if text.strip():
                    lines.append(text.strip())

    return clean_text("\n".join(lines))

In [31]:
doctr_results = load_existing_results(DOCTR_OUT_PATH)
doctr_done_ids = {rec["source_image"] for rec in doctr_results if "source_image" in rec}

print("Already saved:", len(doctr_done_ids))
print("Remaining:", len([p for p in filtered_image_paths if p.name not in doctr_done_ids]))

Already saved: 0
Remaining: 83


In [32]:
for idx, img_path in enumerate(filtered_image_paths, start=1):
    if img_path.name in doctr_done_ids:
        continue

    print(f"[{idx}/{len(filtered_image_paths)}] DocTR {img_path.name}")

    try:
        raw_text = doctr_extract_text(img_path)

        rec = {
            "source_image": img_path.name,
            "ocr_text": raw_text,
        }

    except Exception as e:
        rec = {
            "source_image": img_path.name,
            "ocr_text": "",
            "error": str(e),
        }

    doctr_results.append(rec)
    doctr_done_ids.add(img_path.name)

    save_results(doctr_results, DOCTR_OUT_PATH)
    print("Saved:", img_path.name, "| total saved:", len(doctr_results))

[1/83] DocTR 0.png
Saved: 0.png | total saved: 1
[2/83] DocTR 1.png
Saved: 1.png | total saved: 2
[3/83] DocTR 2.png
Saved: 2.png | total saved: 3
[4/83] DocTR 3.png
Saved: 3.png | total saved: 4
[5/83] DocTR 4.png
Saved: 4.png | total saved: 5
[6/83] DocTR 6.png
Saved: 6.png | total saved: 6
[7/83] DocTR 7.png
Saved: 7.png | total saved: 7
[8/83] DocTR 8.png
Saved: 8.png | total saved: 8
[9/83] DocTR 9.png
Saved: 9.png | total saved: 9
[10/83] DocTR 10.png
Saved: 10.png | total saved: 10
[11/83] DocTR 11.png
Saved: 11.png | total saved: 11
[12/83] DocTR 12.png
Saved: 12.png | total saved: 12
[13/83] DocTR 13.png
Saved: 13.png | total saved: 13
[14/83] DocTR 15.png
Saved: 15.png | total saved: 14
[15/83] DocTR 16.png
Saved: 16.png | total saved: 15
[16/83] DocTR 17.png
Saved: 17.png | total saved: 16
[17/83] DocTR 18.png
Saved: 18.png | total saved: 17
[18/83] DocTR 19.png
Saved: 19.png | total saved: 18
[19/83] DocTR 20.png
Saved: 20.png | total saved: 19
[20/83] DocTR 21.png
Saved: 2

In [33]:
print("Final DocTR records:", len(doctr_results))
doctr_results[:2]

Final DocTR records: 83


[{'source_image': '0.png',
  'ocr_text': '-\n901016\nTICKET CP\n2\n60,000\nC0.000\nTOTAL DISC $\n-60.000\nTAX\n5.455\nSubtotal\nG0.000\nTOTAL\n(Qty\n2.00\n60.000\n-\n- EDC CIMB NIAGA No: XX7730\n60.000\nCICTONER THEA'},
 {'source_image': '1.png',
  'ocr_text': 'J.STB\nPROMO\n17500\nY.B.BAT\n46000\nY.BASO\nPROM\n27500\nTOTAL\n91000\nCASH\n91000\nEe\na'}]

### OpenOCR

In [34]:
from openocr import OpenOCR

In [35]:
openocr_model = OpenOCR()

[2026/04/21 00:12:08] openocr_unified INFO: Initializing OpenOCR with task: ocr
[2026/04/21 00:12:09] openrec INFO: Model already exists at: /root/.cache/openocr/openocr_det_model.onnx
[2026/04/21 00:12:09] openrec INFO: Model already exists at: /root/.cache/openocr/openocr_rec_model.onnx
[2026/04/21 00:12:09] openocr_unified INFO: ✅ OpenOCR initialized successfully for task: ocr


In [36]:
OPENOCR_OUT_PATH = Path("/root/lorex-ocr-llm-experiments/results/cord_ocr/openocr_ocr_cord_test_filtered.json")
print(OPENOCR_OUT_PATH)

/root/lorex-ocr-llm-experiments/results/cord_ocr/openocr_ocr_cord_test_filtered.json


In [45]:
def openocr_extract_text(img_path):
    raw = openocr_model(str(img_path))

    # raw is a tuple: (list_of_result_strings, timing_info)
    if not raw or not isinstance(raw, tuple) or len(raw) == 0:
        return ""

    result_lines = raw[0]
    if not result_lines or not isinstance(result_lines, list):
        return ""

    first = result_lines[0]
    if "\t" not in first:
        return ""

    _, json_part = first.split("\t", 1)
    items = json.loads(json_part)

    texts = []
    for item in items:
        txt = item.get("transcription")
        if txt and txt.strip():
            texts.append(txt.strip())

    return clean_text("\n".join(texts))

In [46]:
if OPENOCR_OUT_PATH.exists():
    OPENOCR_OUT_PATH.unlink()

In [47]:
test_img = filtered_image_paths[0]
txt = openocr_extract_text(test_img)
print(txt)

[2026/04/21 00:54:10] openrec INFO: Processing 1/1: /root/lorex-ocr-llm-experiments/data/cord_test/images/0.png
[2026/04/21 00:54:11] openrec INFO: Results: [{'transcription': '···', 'points': [[148, 342], [198, 342], [198, 356], [148, 356]], 'score': 0.5986440777778625}, {'transcription': '901016', 'points': [[41, 391], [91, 389], [92, 408], [42, 410]], 'score': 0.998011589050293}, {'transcription': '-TICKEI CP', 'points': [[146, 387], [220, 383], [221, 405], [147, 408]], 'score': 0.8819629549980164}, {'transcription': '2', 'points': [[88, 415], [101, 415], [101, 432], [88, 432]], 'score': 0.9990806579589844}, {'transcription': '一', 'points': [[102, 417], [114, 417], [114, 432], [102, 432]], 'score': 0.8257032036781311}, {'transcription': '60.000', 'points': [[176, 410], [221, 410], [221, 429], [176, 429]], 'score': 0.977283775806427}, {'transcription': '80.000', 'points': [[301, 408], [350, 406], [351, 428], [303, 431]], 'score': 0.9011921882629395}, {'transcription': 'TOTAL DISC S',

In [48]:
openocr_results = load_existing_results(OPENOCR_OUT_PATH)
openocr_done_ids = {rec["source_image"] for rec in openocr_results if "source_image" in rec}

print("Already saved:", len(openocr_done_ids))
print("Remaining:", len([p for p in filtered_image_paths if p.name not in openocr_done_ids]))

for idx, img_path in enumerate(filtered_image_paths, start=1):
    if img_path.name in openocr_done_ids:
        continue

    print(f"[{idx}/{len(filtered_image_paths)}] OpenOCR {img_path.name}")

    try:
        raw_text = openocr_extract_text(img_path)

        rec = {
            "source_image": img_path.name,
            "ocr_text": raw_text,
        }

    except Exception as e:
        rec = {
            "source_image": img_path.name,
            "ocr_text": "",
            "error": str(e),
        }

    openocr_results.append(rec)
    openocr_done_ids.add(img_path.name)

    save_results(openocr_results, OPENOCR_OUT_PATH)
    print("Saved:", img_path.name, "| total saved:", len(openocr_results))

Already saved: 0
Remaining: 83
[1/83] OpenOCR 0.png
[2026/04/21 00:54:24] openrec INFO: Processing 1/1: /root/lorex-ocr-llm-experiments/data/cord_test/images/0.png
[2026/04/21 00:54:25] openrec INFO: Results: [{'transcription': '···', 'points': [[148, 342], [198, 342], [198, 356], [148, 356]], 'score': 0.5986440777778625}, {'transcription': '901016', 'points': [[41, 391], [91, 389], [92, 408], [42, 410]], 'score': 0.998011589050293}, {'transcription': '-TICKEI CP', 'points': [[146, 387], [220, 383], [221, 405], [147, 408]], 'score': 0.8819629549980164}, {'transcription': '2', 'points': [[88, 415], [101, 415], [101, 432], [88, 432]], 'score': 0.9990806579589844}, {'transcription': '一', 'points': [[102, 417], [114, 417], [114, 432], [102, 432]], 'score': 0.8257032036781311}, {'transcription': '60.000', 'points': [[176, 410], [221, 410], [221, 429], [176, 429]], 'score': 0.977283775806427}, {'transcription': '80.000', 'points': [[301, 408], [350, 406], [351, 428], [303, 431]], 'score': 0.